In [1]:

# CineMatch — TMDB + IMDb FAISS Update Pipeline
# ==============================================
# Fetches movies from TMDB Discover API in a fixed date window  enriches them with IMDb ratings from title.ratings.tsv,
# creates movieDoc text, encodes with BGE-M3, and upserts vectors into FAISS.

# Workflow:
#     1. Fetch movies via TMDB Discover API (all target languages)
#     2. Fetch full movie details (includes imdb_id + metadata)
#     3. Merge imdb_rating/imdb_votes from Data/title.ratings.tsv
#     4. Derive best_rating / best_votes and build movieDoc
#     5. Upsert vectors into FAISS with add_with_ids()
#     6. Update catalog and manifest

# Usage:
#     python update_faiss.py
    # OR paste into Colab cells
    # Requires: TMDB_BEARER_TOKEN in .env or environment


#use A100 or reduce batch size in main to avoid ood.


!pip install sentence-transformers faiss-gpu-cu12 pandas numpy requests python-dotenv

from __future__ import annotations

import json
import os
import time
from datetime import date
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import requests
import torch
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# ━━━━━━━━━━━━━  CONFIG  ━━━━━━━━━━━━━━

MODEL_ID = "BAAI/bge-m3"
TARGET_LANGS = ["en", "te", "hi", "ta", "ml", "ko", "ja", "es", "fr", "de", "it", "pt", "zh", "tw", "ar"]
FETCH_START_DATE = "2022-01-01"

# Encoding
ENCODE_BATCH = 64

# API
MAX_DISCOVER_PAGES = 500  # per language
DETAIL_BATCH_LOG   = 100  # print progress every N details
IMDB_CHUNK_SIZE    = 1_000_000

# ━━━━━━━━━━━━━  PATHS  ━━━━━━━━━━━━━━━

def detect_paths() -> dict:
    """Auto-detect runtime and resolve paths."""
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive/cinematch")
        print("Runtime: Colab")
    except ImportError:
        hpc = Path("/blue/egn6933/nagabhairava.r")
        if hpc.exists():
            base = hpc
            print("Runtime: HPC")
        else:
            here = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
            for candidate in [here, *here.parents]:
                if (candidate / "Data").exists() and (candidate / "src").exists():
                    base = candidate / "Data"
                    break
            else:
                base = Path.cwd() / "Data"
            print("Runtime: Local")

    out = base / "outputs" / "tmdb" / "bge"
    source_catalog = base / "Data" / "tmdb_semantic_catalog_alllangs_with_new_movies.csv"
    updated_catalog = source_catalog  # Overwrite the old catalog
    catalog_in = updated_catalog if updated_catalog.exists() else source_catalog

    imdb_ratings_tsv = base / "Data" / "title.ratings.tsv"

    return {
        "base":       base,
        "source_catalog": source_catalog,
        "updated_catalog": updated_catalog,
        "catalog_in": catalog_in,
        "out_dir":    out,
        "emb_mmap":   out / "tmdb_bge_m3_embeddings.float32.mmap",
        "checkpoint": out / "tmdb_bge_m3_checkpoint.json",
        "faiss":      out / "tmdb_bge_m3_flatip.faiss",
        "manifest":   out / "tmdb_bge_m3_build_manifest.json",
        "imdb_ratings_tsv": imdb_ratings_tsv,
    }

In [2]:

# ━━━━━━━━━━━━━━━━━━━━  TMDB API  ━━━━━━━━━━━━━━━━━━━━━

def setup_tmdb_auth() -> dict:
    """Load TMDB bearer token from environment or Colab secrets."""
    load_dotenv()
    token = os.environ.get("TMDB_BEARER_TOKEN")
    if not token:
        # Try alternate env var names
        token = os.environ.get("TMDB_API_KEY")

    # If not found in env, try Colab secrets
    if not token:
        try:
            from google.colab import userdata
            try:
                token = userdata.get("TMDB_BEARER_TOKEN")
            except userdata.SecretNotFoundError:
                token = userdata.get("TMDB_API_KEY")
        except (ImportError, Exception):
            pass

    assert token, (
        "Set TMDB_BEARER_TOKEN in your .env file, environment, or Colab Secrets.\n"
        "Get one at https://www.themoviedb.org/settings/api"
    )
    return {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }


TMDB_BASE = "https://api.themoviedb.org/3"


def tmdb_get(url: str, headers: dict, params: dict | None = None,
             max_retries: int = 5, sleep_base: float = 1.0) -> dict:
    """TMDB API request with retry and rate-limit handling."""
    for attempt in range(max_retries):
        r = requests.get(url, headers=headers, params=params, timeout=30)

        if r.status_code == 200:
            return r.json()

        if r.status_code == 429:
            wait = float(r.headers.get("Retry-After", sleep_base * (2 ** attempt)))
            print(f"  [429] Rate limited. Sleeping {wait:.1f}s...")
            time.sleep(wait)
            continue

        if r.status_code in (500, 502, 503, 504):
            wait = sleep_base * (2 ** attempt)
            print(f"  [{r.status_code}] Server error. Sleeping {wait:.1f}s...")
            time.sleep(wait)
            continue

        raise RuntimeError(f"TMDB request failed: {r.status_code} {r.text}")

    raise RuntimeError("Max retries exceeded")


def discover_movies(headers: dict, lang_code: str,
                    start_date: str, end_date: str | None = None,
                    page: int = 1, include_adult: bool = True) -> dict:
    """Call TMDB /discover/movie for one language and date window."""
    if end_date is None:
        end_date = date.today().isoformat()

    return tmdb_get(
        f"{TMDB_BASE}/discover/movie",
        headers=headers,
        params={
            "include_adult": str(include_adult).lower(),
            "include_video": "false",
            "sort_by": "vote_count.desc",
            "page": page,
            "primary_release_date.gte": start_date,
            "primary_release_date.lte": end_date,
            "with_original_language": lang_code,
            "language": "en-US",
        },
    )


def fetch_discover_all(headers: dict, lang_code: str, start_date: str,
                       end_date: str | None = None,
                       max_pages: int = MAX_DISCOVER_PAGES) -> list[dict]:
    """Paginate through all discover results for one language (chunked)."""
    from datetime import datetime, timedelta
    if end_date is None:
        end_date = date.today().isoformat()

    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    results = []
    current_start = start_dt
    while current_start <= end_dt:
        current_end = current_start + timedelta(days=30)
        if current_end > end_dt:
            current_end = end_dt
        s_date = current_start.strftime("%Y-%m-%d")
        e_date = current_end.strftime("%Y-%m-%d")
        page = 1
        while page <= max_pages:
            data = discover_movies(headers, lang_code, s_date, e_date, page)
            batch = data.get("results", [])
            if not batch:
                break
            results.extend(batch)
            if page >= data.get("total_pages", page):
                break
            page += 1
        current_start = current_end + timedelta(days=1)
    return results


def fetch_movie_details(headers: dict, tmdb_id: int) -> dict:
    """Fetch full movie details including keywords."""
    return tmdb_get(
        f"{TMDB_BASE}/movie/{tmdb_id}",
        headers=headers,
        params={
            "language": "en-US",
            "include_adult": "true",
            "append_to_response": "keywords",
        },
    )


In [3]:

# ━━━━━━━━━━━━━━━━━━  MOVIEDOC BUILDER  ━━━━━━━━━━━━━━━

def extract_keyword_names(keywords_obj) -> list[str]:
    """Extract keyword names from TMDB keywords response."""
    if isinstance(keywords_obj, dict):
        items = keywords_obj.get("keywords", []) or keywords_obj.get("results", [])
    elif isinstance(keywords_obj, list):
        items = keywords_obj
    else:
        items = []
    return [k.get("name", "").strip() for k in items
            if isinstance(k, dict) and k.get("name")]


def extract_spoken_language_names(spoken_obj) -> list[str]:
    """Extract spoken language names from TMDB spoken_languages field."""
    if isinstance(spoken_obj, list):
        return [
            (s.get("english_name") or s.get("name") or "").strip()
            for s in spoken_obj
            if isinstance(s, dict) and (s.get("english_name") or s.get("name"))
        ]
    return []


def load_imdb_ratings_subset(
    ratings_tsv: Path,
    imdb_ids: set[str],
    chunk_size: int = IMDB_CHUNK_SIZE,
) -> pd.DataFrame:
    """Load a subset of IMDb ratings keyed by tconst from title.ratings.tsv."""
    if not imdb_ids:
        return pd.DataFrame(columns=["tconst", "imdb_rating", "imdb_votes"])

    if not ratings_tsv.exists():
        print(f"  IMDb ratings TSV not found: {ratings_tsv}")
        return pd.DataFrame(columns=["tconst", "imdb_rating", "imdb_votes"])

    print(f"  Loading IMDb ratings from: {ratings_tsv}")
    matches = []
    chunks = pd.read_csv(
        ratings_tsv,
        sep="\t",
        usecols=["tconst", "averageRating", "numVotes"],
        dtype={"tconst": "string", "averageRating": "float32", "numVotes": "Int64"},
        chunksize=chunk_size,
        low_memory=False,
    )

    for i, chunk in enumerate(chunks, 1):
        picked = chunk[chunk["tconst"].isin(imdb_ids)]
        if not picked.empty:
            matches.append(picked)
        if i % 10 == 0:
            print(f"  IMDb chunks scanned: {i:,}")

    if not matches:
        return pd.DataFrame(columns=["tconst", "imdb_rating", "imdb_votes"])

    imdb = pd.concat(matches, ignore_index=True)
    imdb = imdb.rename(columns={"averageRating": "imdb_rating", "numVotes": "imdb_votes"})
    imdb["imdb_rating"] = pd.to_numeric(imdb["imdb_rating"], errors="coerce")
    imdb["imdb_votes"] = pd.to_numeric(imdb["imdb_votes"], errors="coerce").fillna(0).astype(int)
    imdb = imdb.dropna(subset=["tconst"]).drop_duplicates(subset=["tconst"])
    return imdb


def build_moviedoc(row: dict) -> str:
    """Build movieDoc string from TMDB detail response."""
    title    = str(row.get("title", "") or "").strip()
    original_title = str(row.get("original_title", "") or "").strip()
    overview = str(row.get("overview", "") or "").strip()
    tagline  = str(row.get("tagline", "") or "").strip()
    lang     = str(row.get("original_language", "") or "").strip()
    imdb_id  = str(row.get("imdb_id", "") or "").strip()

    release_date = str(row.get("release_date", "") or "").strip()
    year = str(row.get("year", "") or "").strip()
    if not year:
        year = release_date[:4] if release_date and len(release_date) >= 4 and release_date[:4].isdigit() else ""

    vote_average = row.get("vote_average")
    vote_count   = row.get("vote_count")
    imdb_rating  = row.get("imdb_rating")
    imdb_votes   = row.get("imdb_votes")
    best_rating  = row.get("best_rating")
    best_votes   = row.get("best_votes")
    popularity   = row.get("popularity")

    genres = row.get("genres", [])
    if isinstance(genres, list):
        genres_list = [g.get("name") for g in genres if isinstance(g, dict) and g.get("name")]
    else:
        genres_list = [x.strip() for x in str(genres).split(",") if x.strip()] if pd.notna(genres) else []

    spoken_list = extract_spoken_language_names(row.get("spoken_languages", []))

    keywords_list = extract_keyword_names(row.get("keywords", {}))

    lines = [
        f"Title: {title}",
        f"Original title: {original_title}" if original_title and original_title != title else None,
        f"Release date: {release_date}" if release_date else None,
        f"Year: {year}" if year else None,
        f"IMDb id: {imdb_id}" if imdb_id else None,
        f"Original language: {lang}" if lang else None,
        f"Spoken languages: {', '.join(spoken_list[:5])}" if spoken_list else None,
        f"Genres: {', '.join(genres_list[:6])}" if genres_list else None,
        f"IMDb rating: {float(imdb_rating):.2f}" if pd.notna(imdb_rating) else None,
        f"IMDb votes: {int(imdb_votes)}" if pd.notna(imdb_votes) else None,
        f"Best rating: {float(best_rating):.2f}" if pd.notna(best_rating) else None,
        f"Best votes: {int(best_votes)}" if pd.notna(best_votes) else None,
        f"Vote average: {float(vote_average):.2f}" if pd.notna(vote_average) else None,
        f"Vote count: {int(vote_count)}" if pd.notna(vote_count) else None,
        f"Popularity: {float(popularity):.2f}" if pd.notna(popularity) else None,
        f"Keywords: {', '.join(keywords_list[:12])}" if keywords_list else None,
        f"Tagline: {tagline}" if tagline else None,
        f"Plot: {overview}" if overview else None,
    ]
    return "\n".join([x for x in lines if x])

In [4]:
import torch
import os
os.environ["PYTORCH_NO_CUDA_MEMORY_CACHING"] = "1"

In [5]:

# ━━━━━━━━━━━━━━━━━━━━━  MAIN  ━━━━━━━━━━━━━━━━━━━━━━━

def main():
    import concurrent.futures
    t_start = time.time()
    paths = detect_paths()
    paths["out_dir"].mkdir(parents=True, exist_ok=True)
    paths["updated_catalog"].parent.mkdir(parents=True, exist_ok=True)
    headers = setup_tmdb_auth()

    print(f"\n{'─'*60}")
    print(" Determining update window")
    print(f"{'─'*60}")

    fetch_start = FETCH_START_DATE
    today = date.today().isoformat()
    print(f"  Fetching new movies from {fetch_start} to {today}")

    # ── Fetch new movies from TMDB API ────────────────────
    print(f"\n{'─'*60}")
    print(" Detching new movies from TMDB Discover API")
    print(f"{'─'*60}")

    # Load existing IDs to avoid duplicates
    existing_ids = set()
    if paths["catalog_in"].exists():
        df_existing = pd.read_csv(paths["catalog_in"], usecols=["id"], low_memory=False)
        df_existing["id"] = pd.to_numeric(df_existing["id"], errors="coerce")
        existing_ids = set(df_existing["id"].dropna().astype(int).tolist())
        print(f"  Existing catalog IDs: {len(existing_ids):,}")

    # Also check FAISS index for already-indexed IDs
    if paths["faiss"].exists():
        idx = faiss.read_index(str(paths["faiss"]))
        print(f"  Existing FAISS vectors: {idx.ntotal:,}")
        # Get all IDs from IndexIDMap2 — these are the TMDB ids already indexed
        if hasattr(idx, "id_map"):
            faiss_ids = set(faiss.vector_to_array(idx.id_map).tolist())
            existing_ids.update(int(i) for i in faiss_ids if i >= 0)
    else:
        idx = None

    all_discover = []
    for lang in TARGET_LANGS:
        print(f"  Discovering {lang}...", end=" ")
        results = fetch_discover_all(headers, lang, start_date=fetch_start, end_date=today)
        print(f"{len(results)} results")
        all_discover.extend(results)

    # Deduplicate. We process ALL discovered IDs to overwrite any outdated existing data.
    discover_df = pd.DataFrame(all_discover).drop_duplicates(subset=["id"])
    discover_df["id"] = pd.to_numeric(discover_df["id"], errors="coerce")
    active_ids = [
        int(x) for x in discover_df["id"].dropna().unique()
    ]
    print(f"\n  Discover unique (active window): {len(discover_df):,}")

    if not active_ids:
        print("\n  No movies found in update window. Exiting.")
        return

    # ──  Fetch movie details ───────────────────────────────
    print(f"\n{'─'*60}")
    print(f" Fetching details for {len(active_ids):,} active movies concurrently")
    print(f"{'─'*60}")

    details_list = []

    # Use ThreadPoolExecutor for max 40 parallel connections
    with concurrent.futures.ThreadPoolExecutor(max_workers=40) as executor:
        future_to_id = {}
        for tid in active_ids:
            future = executor.submit(fetch_movie_details, headers, tid)
            future_to_id[future] = tid

        for i, future in enumerate(concurrent.futures.as_completed(future_to_id), 1):
            tid = future_to_id[future]
            try:
                details_list.append(future.result())
            except RuntimeError as e:
                print(f"  Skipped {tid}: {e}")
            if i % DETAIL_BATCH_LOG == 0:
                print(f"  Fetched {i:,}/{len(active_ids):,}")

    if not details_list:
        print("  No details fetched. Exiting.")
        return

    print(f"  Fetched {len(details_list):,} movie details")

    # Merge IMDb ratings from title.ratings.tsv
    new_movies = pd.DataFrame(details_list)
    new_movies["id"] = pd.to_numeric(new_movies["id"], errors="coerce")
    new_movies = new_movies.dropna(subset=["id"])
    new_movies["id"] = new_movies["id"].astype(int)

    # Build year for downstream catalog and movieDoc.
    new_movies["release_date"] = new_movies["release_date"].fillna("").astype(str)
    new_movies["year"] = new_movies["release_date"].str.slice(0, 4)
    new_movies.loc[~new_movies["year"].str.match(r"^\d{4}$", na=False), "year"] = ""

    imdb_ids = set(
        new_movies["imdb_id"].dropna().astype(str).str.strip().loc[lambda s: s.str.startswith("tt")].tolist()
    )
    print(f"  Movies with imdb_id: {len(imdb_ids):,}")
    imdb_subset = load_imdb_ratings_subset(paths["imdb_ratings_tsv"], imdb_ids)

    if not imdb_subset.empty:
        new_movies = new_movies.merge(imdb_subset, left_on="imdb_id", right_on="tconst", how="left")
        new_movies = new_movies.drop(columns=["tconst"], errors="ignore")
        print(f"  IMDb matches: {new_movies['imdb_rating'].notna().sum():,} / {len(new_movies):,}")
    else:
        new_movies["imdb_rating"] = np.nan
        new_movies["imdb_votes"] = 0
        print("  IMDb matches: 0 (no matching records found in title.ratings.tsv)")

    # Derive best_* columns, preferring IMDb when available.
    new_movies["vote_average"] = pd.to_numeric(new_movies["vote_average"], errors="coerce")
    new_movies["vote_count"] = pd.to_numeric(new_movies["vote_count"], errors="coerce").fillna(0)
    new_movies["imdb_rating"] = pd.to_numeric(new_movies["imdb_rating"], errors="coerce")
    new_movies["imdb_votes"] = pd.to_numeric(new_movies["imdb_votes"], errors="coerce").fillna(0).astype(int)
    new_movies["best_rating"] = new_movies["imdb_rating"].fillna(new_movies["vote_average"])
    new_movies["best_votes"] = new_movies["imdb_votes"].where(new_movies["imdb_votes"] > 0, new_movies["vote_count"])
    new_movies["best_votes"] = pd.to_numeric(new_movies["best_votes"], errors="coerce").fillna(0).astype(int)

    # Build movieDocs with TMDB + IMDb signals.
    new_movies["movieDoc"] = new_movies.apply(
        lambda row: build_moviedoc(row.to_dict()), axis=1
    )

    # Filter embeddable
    mask = new_movies["movieDoc"].str.contains("Plot:", na=False) & (new_movies["movieDoc"].str.len() >= 80)
    new_embed = new_movies.loc[mask, ["id", "movieDoc", "runtime", "status", "title"]].copy()
    print(f"  Embeddable new movies: {len(new_embed):,}")

    if new_embed.empty:
        print(" No embeddable new movies. Exiting.")
        return

    # ── Update catalog CSV ────────────────────────────────
    print(f"\n{'─'*60}")
    print("  Updating catalog CSV")
    print(f"{'─'*60}")

    # Prepare new rows for catalog
    keep_cols = [
        "id", "title", "release_date", "backdrop_path", "imdb_id", "original_language",
        "original_title", "overview", "popularity", "poster_path", "tagline",
        "genres", "spoken_languages", "keywords", "year",
        "vote_average", "vote_count", "imdb_rating", "imdb_votes", "best_rating", "best_votes",
        "movieDoc", "runtime", "status",
    ]
    for c in keep_cols:
        if c not in new_movies.columns:
            new_movies[c] = np.nan

    # Flatten genres and keywords for CSV storage
    def flatten_genres(g):
        if isinstance(g, list):
            return ", ".join(x.get("name", "") for x in g if isinstance(x, dict))
        return str(g) if pd.notna(g) else ""

    def flatten_keywords(k):
        names = extract_keyword_names(k)
        return ", ".join(names)

    def flatten_spoken_languages(s):
        names = extract_spoken_language_names(s)
        return ", ".join(names)

    new_for_catalog = new_movies[keep_cols].copy()
    new_for_catalog["genres"] = new_for_catalog["genres"].apply(flatten_genres)
    new_for_catalog["spoken_languages"] = new_for_catalog["spoken_languages"].apply(flatten_spoken_languages)
    new_for_catalog["keywords"] = new_for_catalog["keywords"].apply(flatten_keywords)

    # --- Print a sample row ---
    if not new_for_catalog.empty:
        print(f"\n  [Sample] First row of processed data:")
        sample_row = new_for_catalog.iloc[5].to_dict()
        for k, v in sample_row.items():
            val_str = str(v)
            if len(val_str) > 300:
                val_str = val_str[:147] + "..."
            print(f"    {k}: {val_str}")
        print("")

    # Replace ghost entries in existing catalog by title before resolving by id
    titles_to_replace = set(new_for_catalog["title"].dropna().unique())
    idx_to_remove = []

    if paths["catalog_in"].exists():
        df_catalog = pd.read_csv(paths["catalog_in"], low_memory=False)

        # Upsert by id while preserving any existing columns not present in new_for_catalog.
        df_catalog["id"] = pd.to_numeric(df_catalog["id"], errors="coerce")
        new_for_catalog["id"] = pd.to_numeric(new_for_catalog["id"], errors="coerce")
        df_catalog = df_catalog.dropna(subset=["id"])
        new_for_catalog = new_for_catalog.dropna(subset=["id"])

        df_catalog["id"] = df_catalog["id"].astype(int)
        new_for_catalog["id"] = new_for_catalog["id"].astype(int)



        df_catalog = df_catalog.drop_duplicates(subset=["id"], keep="last").set_index("id")
        new_for_catalog = new_for_catalog.drop_duplicates(subset=["id"], keep="last").set_index("id")

        # Add missing target columns to existing catalog before update.
        for col in new_for_catalog.columns:
            if col not in df_catalog.columns:
                df_catalog[col] = np.nan

        df_catalog.update(new_for_catalog)
        new_ids = new_for_catalog.index.difference(df_catalog.index)
        if len(new_ids) > 0:
            df_catalog = pd.concat([df_catalog, new_for_catalog.loc[new_ids]], axis=0)

        df_catalog = df_catalog.reset_index()
    else:
        df_catalog = new_for_catalog

    df_catalog.to_csv(paths["updated_catalog"], index=False)
    print(f" Updated catalog: {len(df_catalog):,} total rows → {paths['updated_catalog'].name}")

    # ─ Encode new movies ─────────────────────────────────
    print(f"\n{'─'*60}")
    print(f" Encoding {len(new_embed):,} new movies with {MODEL_ID}")
    print(f"{'─'*60}")

    if torch.cuda.is_available():
        device = "cuda"
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"

    model = SentenceTransformer(MODEL_ID, device=device)
    emb_dim = model.get_sentence_embedding_dimension()
    if emb_dim is None:
        probe = model.encode(["dim probe"], convert_to_numpy=True, normalize_embeddings=True)
        emb_dim = int(probe.shape[1])

    print(f"  Device: {device}  |  Dim: {emb_dim}")

    texts = new_embed["movieDoc"].tolist()
    new_ids_arr = new_embed["id"].to_numpy(dtype=np.int64)

    with torch.no_grad():
        vectors = model.encode(
            texts,
            batch_size=ENCODE_BATCH,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

    # Double normalize
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    vectors = vectors / norms

    print(f" Encoded {len(vectors):,} vectors")

    # ── Update FAISS index ────────────────────────────────
    print(f"\n{'─'*60}")
    print(" Updating FAISS index")
    print(f"{'─'*60}")

    if paths["faiss"].exists():
        index = faiss.read_index(str(paths["faiss"]))
        old_total = index.ntotal
        print(f"  Loaded existing index: {old_total:,} vectors")

        # Remove old vectors for overwritten IDs AND any ghost entries we found by title
        rem_ids = [i for i in new_ids_arr if i in existing_ids]
        rem_ids.extend(idx_to_remove)
        rem_ids = np.array(rem_ids, dtype=np.int64)

        if len(rem_ids) > 0:
            try:
                index.remove_ids(rem_ids)
                print(f"  Removed {len(rem_ids):,} old/ghost vectors for overwrite.")
            except AttributeError:
                print("  ⚠ Index doesn't support remove_ids. Overwrite appended as duplicates.")
    else:
        base_index = faiss.IndexFlatIP(emb_dim)
        index = faiss.IndexIDMap2(base_index)
        old_total = 0
        print(f"  Created new index (dim={emb_dim})")

    index.add_with_ids(vectors, new_ids_arr)
    faiss.write_index(index, str(paths["faiss"]))
    print(f"Added/Updated {len(vectors):,} vectors. Total: {index.ntotal:,}")

    # ── Update memmap (append) ────────────────────────────
    # Note: For the incremental case we also need to grow the memmap.
    # The cleanest approach is to track the delta separately.
    delta_mmap_path = paths["out_dir"] / f"delta_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}.npy"
    np.save(delta_mmap_path, vectors)
    delta_ids_path = delta_mmap_path.with_suffix(".ids.npy")
    np.save(delta_ids_path, new_ids_arr)
    print(f" Delta saved: {delta_mmap_path.name} ({len(vectors):,} vectors)")

    # ── Update manifest ───────────────────────────────────
    manifest = {}
    if paths["manifest"].exists():
        manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))

    update_record = {
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "fetch_window": f"{fetch_start} → {today}",
        "movies_discovered": len(active_ids),
        "movies_embedded_and_updated": len(vectors),
        "faiss_total_after": int(index.ntotal),
        "delta_file": str(delta_mmap_path),
    }

    if "updates" not in manifest:
        manifest["updates"] = []
    manifest["updates"].append(update_record)

    # Update top-level timestamp
    if "embedding" in manifest:
        manifest["embedding"]["timestamp_utc"] = pd.Timestamp.utcnow().isoformat()
        manifest["embedding"]["faiss"]["ntotal"] = int(index.ntotal)

    paths["manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f" Manifest updated: {paths['manifest'].name}")

    total_time = time.time() - t_start
    print(f"\n{'═'*60}")
    print(f"  FAISS UPDATE DONE — {len(vectors):,} movies updated in {total_time:.1f}s")
    print(f"{'═'*60}")


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: Colab

────────────────────────────────────────────────────────────
 Determining update window
────────────────────────────────────────────────────────────
  Fetching new movies from 2026-03-01 to 2026-08-24

────────────────────────────────────────────────────────────
 Detching new movies from TMDB Discover API
────────────────────────────────────────────────────────────
  Existing catalog IDs: 1,320,040
  Existing FAISS vectors: 1,316,607
  Discovering en... 9999 results
  Discovering te... 103 results
  Discovering hi... 154 results
  Discovering ta... 94 results
  Discovering ml... 78 results
  Discovering ko... 469 results
  Discovering ja... 618 results
  Discovering es... 1627 results
  Discovering fr... 1791 results
  Discovering de... 640 results
  Discovering it... 447 results
  Discovering pt... 1099 results
  Discovering zh... 590 results

/tmp/ipykernel_15488/1229771566.py:221: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1988.0 1986.0 1995.0 ... '2026' '2026' nan]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_catalog.update(new_for_catalog)


 Updated catalog: 1,320,303 total rows → tmdb_semantic_catalog_alllangs_with_new_movies.csv

────────────────────────────────────────────────────────────
 Encoding 14,356 new movies with BAAI/bge-m3
────────────────────────────────────────────────────────────


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

  Device: cuda  |  Dim: 1024


/tmp/ipykernel_15488/1229771566.py:246: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  emb_dim = model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [8]:
# need to update complete faiss as we added new imdb data

In [7]:
import os
from huggingface_hub import HfApi, login
from google.colab import userdata

paths = detect_paths()

# 1. Authenticate with Hugging Face
try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token)
except Exception as e:
    print("❌ Please add your Hugging Face write token to Colab Secrets as 'HF_TOKEN'.")
    raise e

api = HfApi()
repo_id = "ml8r/cinematch"
repo_type = "dataset"

# Define the files and their destination paths in the HF repo
upload_tasks = [
    {
        "local": str(paths["source_catalog"]),
        "remote": "Data/tmdb_semantic_catalog_alllangs_with_new_movies.csv",
        "msg": "Update TMDB semantic catalog with new IMDb ratings and movieDocs"
    },
    {
        "local": str(paths["out_dir"] / "tmdb_bge_m3_flatip.faiss"),
        "remote": "outputs/tmdb/bge/tmdb_bge_m3_flatip.faiss",
        "msg": "Upload full rebuild FAISS index (BGE-M3)"
    },
    {
        "local": str(paths["manifest"]),
        "remote": "outputs/tmdb/bge/tmdb_bge_m3_build_manifest.json",
        "msg": "Update FAISS build manifest"
    }
]

print(f"\nInitiating upload to dataset: {repo_id}...")

for task in upload_tasks:
    local_path = task["local"]
    remote_path = task["remote"]

    if os.path.exists(local_path):
        print(f"\n⬆️ Uploading: {os.path.basename(local_path)}\n   To: {remote_path}")
        try:
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=remote_path,
                repo_id=repo_id,
                repo_type=repo_type,
                commit_message=task["msg"]
            )
            print("✅ Upload successful.")
        except Exception as e:
            print(f"❌ Failed to upload {local_path}: {e}")
    else:
        print(f"\n⚠️ Warning: Local file not found: {local_path}. Skipping.")

print("\n🎉 Push to Hugging Face complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: Colab

Initiating upload to dataset: ml8r/cinematch...

⬆️ Uploading: tmdb_semantic_catalog_alllangs_with_new_movies.csv
   To: Data/tmdb_semantic_catalog_alllangs_with_new_movies.csv


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...langs_with_new_movies.csv:   0%|          | 2.22MB / 1.18GB            

✅ Upload successful.

⬆️ Uploading: tmdb_bge_m3_flatip.faiss
   To: outputs/tmdb/bge/tmdb_bge_m3_flatip.faiss


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../tmdb_bge_m3_flatip.faiss:   0%|          |  783kB / 5.46GB            

✅ Upload successful.

⬆️ Uploading: tmdb_bge_m3_build_manifest.json
   To: outputs/tmdb/bge/tmdb_bge_m3_build_manifest.json
✅ Upload successful.

🎉 Push to Hugging Face complete!


### Continuation & Recovery
This cell resumes the encoding process. It loads the `updated_catalog` (which successfully saved before the crash) and checks which IDs are missing from the current FAISS index. It encodes only those missing movies with a significantly smaller `ENCODE_BATCH` (64 instead of 2048) to avoid CUDA Out of Memory errors, and uses the updated `get_embedding_dimension()` method.

In [6]:
import pandas as pd
import faiss
import numpy as np
import torch
import time
import json
from sentence_transformers import SentenceTransformer

# 1. Setup
paths = detect_paths()
ENCODE_BATCH = 64  # Reduced batch size to prevent OOM
MODEL_ID = "BAAI/bge-m3"

print("Loading catalog and FAISS index...")

# 2. Load catalog
catalog = pd.read_csv(paths["updated_catalog"], low_memory=False)
catalog["id"] = pd.to_numeric(catalog["id"], errors="coerce")
catalog = catalog.dropna(subset=["id"])
catalog["id"] = catalog["id"].astype(int)

# 3. Load FAISS and find missing IDs
existing_faiss_ids = set()
if paths["faiss"].exists():
    index = faiss.read_index(str(paths["faiss"]))
    if hasattr(index, "id_map"):
        existing_faiss_ids = set(faiss.vector_to_array(index.id_map).tolist())
else:
    index = None

missing_mask = ~catalog["id"].isin(existing_faiss_ids)
valid_mask = catalog["movieDoc"].str.contains("Plot:", na=False) & (catalog["movieDoc"].str.len() >= 80)
new_embed = catalog[missing_mask & valid_mask].copy()

print(f"Found {len(new_embed):,} new embeddable movies missing from FAISS.")

if not new_embed.empty:
    # 4. Initialize Model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(MODEL_ID, device=device)

    # FIXED: using get_embedding_dimension to avoid the warning
    emb_dim = model.get_embedding_dimension()
    print(f"Device: {device}  |  Dim: {emb_dim}  |  Batch Size: {ENCODE_BATCH}")

    texts = new_embed["movieDoc"].tolist()
    new_ids_arr = new_embed["id"].to_numpy(dtype=np.int64)

    # 5. Encode
    with torch.no_grad():
        vectors = model.encode(
            texts,
            batch_size=ENCODE_BATCH,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

    # Double normalize
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    vectors = vectors / norms
    print(f"Encoded {len(vectors):,} vectors.")

    # 6. Update FAISS
    if index is None:
        base_index = faiss.IndexFlatIP(emb_dim)
        index = faiss.IndexIDMap2(base_index)

    index.add_with_ids(vectors, new_ids_arr)
    faiss.write_index(index, str(paths["faiss"]))
    print(f"Added/Updated {len(vectors):,} vectors. Total: {index.ntotal:,}")

    # 7. Quick Manifest Update
    if paths["manifest"].exists():
        manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"))
        if "updates" not in manifest: manifest["updates"] = []
        manifest["updates"].append({
            "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
            "note": "Continuation script append",
            "movies_embedded_and_updated": len(vectors),
            "faiss_total_after": int(index.ntotal)
        })
        paths["manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        print("Manifest updated.")
else:
    print("No new movies need encoding. You are all caught up!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: Colab
Loading catalog and FAISS index...
Found 12,585 new embeddable movies missing from FAISS.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Device: cuda  |  Dim: 1024  |  Batch Size: 64


Batches:   0%|          | 0/197 [00:00<?, ?it/s]

Encoded 12,585 vectors.
Added/Updated 12,585 vectors. Total: 1,329,192
Manifest updated.
